In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import h5py

DATA_PATH = "/kaggle/input/ct-dataset-siddon/ct_dataset_siddon.h5"
SAVE_PATH = "dncnn_siddon_mse.pth"

class CTDataset(Dataset):
    def __init__(self, h5_path, split):
        self.h5_path = h5_path
        self.split = split
        with h5py.File(h5_path, 'r') as f:
            self.length = f[f'{split}/clean'].shape[0]
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            clean = torch.tensor(f[f'{self.split}/clean'][idx])
            noisy = torch.tensor(f[f'{self.split}/noisy'][idx])
        return noisy, clean

class DnCNN(nn.Module):
    def __init__(self, depth=17, n_channels=64, image_channels=1):  # 17 — matches paper
        super().__init__()
        layers = [nn.Conv2d(image_channels, n_channels, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(n_channels, n_channels, 3, padding=1, bias=False),
                       nn.BatchNorm2d(n_channels), nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(n_channels, image_channels, 3, padding=1))
        self.dncnn = nn.Sequential(*layers)
    def forward(self, x):
        return x - self.dncnn(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("Model: DnCNN depth=17, MSE loss only — matches Zhang et al. (2017)")

train_loader = DataLoader(CTDataset(DATA_PATH, 'train'), batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CTDataset(DATA_PATH, 'val'),   batch_size=8, shuffle=False, num_workers=2)

model     = DnCNN(depth=17).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

best_val = float('inf')
print(f"\n{'Epoch':>5} {'Train':>10} {'Val':>10}")
print("-" * 30)

for epoch in range(1, 51):  # 50 epochs 
    model.train()
    train_loss = 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(noisy), clean)  # MSE only —
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            val_loss += nn.functional.mse_loss(model(noisy), clean).item()
    val_loss /= len(val_loader)

    scheduler.step()

    saved = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        saved = '*** SAVED'
    print(f"{epoch:>5} {train_loss:>10.6f} {val_loss:>10.6f} {saved}")

print(f"\nDone! Best val loss: {best_val:.6f}")
print(f"Download {SAVE_PATH} from the files panel on the left.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import h5py


DATA_PATH = "/kaggle/input/datasets/maryamh22/ct-data-siddon/ct_dataset_siddon.h5"
SAVE_PATH = "dncnn_siddon_ssim.pth"

class CTDataset(Dataset):
    def __init__(self, h5_path, split):
        self.h5_path = h5_path
        self.split = split
        with h5py.File(h5_path, 'r') as f:
            self.length = f[f'{split}/clean'].shape[0]
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            clean = torch.tensor(f[f'{self.split}/clean'][idx])
            noisy = torch.tensor(f[f'{self.split}/noisy'][idx])
        return noisy, clean

class DnCNN(nn.Module):
    def __init__(self, depth=17, n_channels=64, image_channels=1):
        super().__init__()
        layers = [nn.Conv2d(image_channels, n_channels, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(n_channels, n_channels, 3, padding=1, bias=False),
                       nn.BatchNorm2d(n_channels), nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(n_channels, image_channels, 3, padding=1))
        self.dncnn = nn.Sequential(*layers)
    def forward(self, x):
        return x - self.dncnn(x)

# ── SSIM loss — same implementation as v2/v3 ──────────────────────────────────

def ssim_loss(pred, target, window_size=11):
    C1, C2 = 0.01**2, 0.03**2
    mu1 = F.avg_pool2d(pred,   window_size, 1, window_size//2)
    mu2 = F.avg_pool2d(target, window_size, 1, window_size//2)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1*mu2
    sigma1_sq = F.avg_pool2d(pred*pred,     window_size, 1, window_size//2) - mu1_sq
    sigma2_sq = F.avg_pool2d(target*target, window_size, 1, window_size//2) - mu2_sq
    sigma12   = F.avg_pool2d(pred*target,   window_size, 1, window_size//2) - mu1_mu2
    ssim_map  = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
    return 1 - ssim_map.mean()

def combined_loss(pred, target, alpha=0.7):
    return alpha * F.mse_loss(pred, target) + (1 - alpha) * ssim_loss(pred, target)

# ── Training ──────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("Model: DnCNN depth=17, MSE+SSIM loss, 50 epochs")

train_loader = DataLoader(CTDataset(DATA_PATH, 'train'), batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CTDataset(DATA_PATH, 'val'),   batch_size=8, shuffle=False, num_workers=2)

model     = DnCNN(depth=17).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

best_val = float('inf')
print(f"\n{'Epoch':>5} {'Train':>10} {'Val':>10}")
print("-" * 30)

for epoch in range(1, 51):  # 50 epochs — isolates loss function change only
    model.train()
    train_loss = 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()
        loss = combined_loss(model(noisy), clean)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            val_loss += combined_loss(model(noisy), clean).item()
    val_loss /= len(val_loader)

    scheduler.step()

    saved = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        saved = '*** SAVED'
    print(f"{epoch:>5} {train_loss:>10.6f} {val_loss:>10.6f} {saved}")

print(f"\nDone! Best val loss: {best_val:.6f}")
print(f"Download {SAVE_PATH} from the files panel on the left.")